# Local notebook setup

This notebook is set up for a local VS Code workflow.

Before running the Python example cells:

1. Install `uv` locally.
2. Start SurrealDB in a separate terminal so `ws://localhost:8000/rpc` is reachable.
3. Run the next install cell once, then restart the notebook kernel.

If you use non-default settings, set `SURREAL_URL`, `SURREAL_USERNAME`, `SURREAL_PASSWORD`, `SURREAL_NAMESPACE`, and `SURREAL_DATABASE` before opening the notebook.


In [3]:
import shutil
import subprocess
import sys

packages = [
    "langchain-surrealdb",
    "surrealdb",
    "langchain-huggingface",
    "sentence-transformers",
    "langchain-community",
]

if not shutil.which("uv"):
    raise RuntimeError(
        "uv is not installed. Install it first, for example with `brew install uv`, then rerun this cell."
    )

command = [
    "uv",
    "pip",
    "install",
    "--python",
    sys.executable,
    "--upgrade",
    *packages,
]

print("Running:", " ".join(command))
subprocess.run(command, check=True)
print("Dependency install complete. Restart the kernel before running the next cell.")

Running: uv pip install --python /Users/eldarutiushev/Projects/Soreal/.venv/bin/python --upgrade langchain-surrealdb surrealdb langchain-huggingface sentence-transformers langchain-community
Dependency install complete. Restart the kernel before running the next cell.


Resolved 71 packages in 1.49s
Audited 71 packages in 2ms


<!-- > **Environment note:** This notebook targets Python 3.10+ and only covers the vector store APIs.
>
> It assumes SurrealDB is already running locally in a separate terminal and that you have restarted the kernel after the dependency install cell.
>
> If you plan to explore graph QA separately, install the optional extra outside this notebook:
>
> ```bash
> uv pip install --python $(which python) "langchain-surrealdb[graph-qa]"
> ``` -->


# SurrealDBVectorStore

This notebook covers how to get started with the SurrealDB vector store.


## Setup

Run this notebook against a local SurrealDB instance.

### 1. Start SurrealDB outside the notebook

If you have the native binary installed:

```bash
surreal start --user root --pass root memory
```

Or run it with Docker:

```bash
docker run --rm --pull always -p 8000:8000 surrealdb/surrealdb:latest start --user root --pass root memory
```

### 2. Install Python dependencies

Run the install cell near the top of this notebook once. It uses `uv` to install packages into the active notebook interpreter.

### 3. Restart the kernel

Restart the kernel after the install cell completes, then continue with initialization.

If you want to use custom settings, export these environment variables before launching the notebook:

```bash
export SURREAL_URL=ws://localhost:8000/rpc
export SURREAL_USERNAME=root
export SURREAL_PASSWORD=root
export SURREAL_NAMESPACE=langchain
export SURREAL_DATABASE=demo
```


## Initialization


In [14]:
import os
import socket
import time
from urllib.parse import urlparse

SURREAL_URL = os.getenv("SURREAL_URL", "ws://localhost:8000/rpc")
SURREAL_USERNAME = os.getenv("SURREAL_USERNAME", "root")
SURREAL_PASSWORD = os.getenv("SURREAL_PASSWORD", "root")
SURREAL_NAMESPACE = os.getenv("SURREAL_NAMESPACE", "langchain")
SURREAL_DATABASE = os.getenv("SURREAL_DATABASE", "demo")

def wait_for_surreal(url: str, timeout: float = 15.0, interval: float = 0.5) -> None:
    parsed = urlparse(url)
    host = parsed.hostname or "localhost"
    port = parsed.port or (443 if parsed.scheme == "wss" else 80)
    deadline = time.time() + timeout
    last_error = None

    while time.time() < deadline:
        try:
            with socket.create_connection((host, port), timeout=2):
                return
        except OSError as exc:
            last_error = exc
            time.sleep(interval)

    raise ConnectionError(
        "Could not reach SurrealDB at "
        f"{url}. Start it in a separate terminal before running this notebook."
    ) from last_error

wait_for_surreal(SURREAL_URL)
print(f"SurrealDB is reachable at {SURREAL_URL}")

SurrealDB is reachable at ws://localhost:8000/rpc


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from surrealdb import Surreal

from langchain_surrealdb import vectorstores as surreal_vectorstores
from langchain_surrealdb.vectorstores import SurrealDBVectorStore

surreal_vectorstores.DEFINE_INDEX = """
    DEFINE INDEX IF NOT EXISTS {index_name}
        ON TABLE {table}
        FIELDS vector
        HNSW DIMENSION {embedding_dimension} DIST COSINE
        CONCURRENTLY;
"""

surreal_vectorstores.SEARCH_QUERY = """
    SELECT
        id,
        text,
        metadata,
        vector,
        vector::similarity::cosine(vector, $vector) AS similarity
    FROM type::table($table)
    WHERE vector <|{k},40|> $vector
    ORDER BY similarity DESC
"""

conn = Surreal(SURREAL_URL)
_ = conn.signin({"username": SURREAL_USERNAME, "password": SURREAL_PASSWORD})
_ = conn.use(SURREAL_NAMESPACE, SURREAL_DATABASE)

model_name = "BAAI/bge-small-en-v1.5"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
 )

vector_store = SurrealDBVectorStore(embeddings, conn)
print(
    f"Connected to SurrealDB at {SURREAL_URL} using "
    f"namespace={SURREAL_NAMESPACE} database={SURREAL_DATABASE}"
)

Connected to SurrealDB at ws://localhost:8000/rpc using namespace=langchain database=demo


## Manage vector store

### Add items to vector store


In [15]:
from langchain_core.documents import Document

_url = "https://surrealdb.com"
d1 = Document(page_content="foo", metadata={"source": _url})
d2 = Document(page_content="SurrealDB", metadata={"source": _url})
d3 = Document(page_content="bar", metadata={"source": _url})
d4 = Document(page_content="this is surreal", metadata={"source": _url})

_ = vector_store.add_documents(documents=[d1, d2, d3, d4], ids=["1", "2", "3", "4"])

### Update items in vector store


In [16]:
updated_document = Document(
    page_content="zar", metadata={"source": "https://example.com"}
)

_ = vector_store.add_documents(documents=[updated_document], ids=["3"])

### Delete items from vector store


In [17]:
vector_store.delete(ids=["3"])

## Query vector store

Once your vector store has been created and the relevant documents have been added you will most likely wish to query it during the running of your chain or agent.

### Query directly

Performing a simple similarity search can be done as follows:


In [ ]:
results = [
    doc
    for doc in vector_store.similarity_search(query="surreal", k=4)
    if doc.metadata.get("source") == "https://surrealdb.com"
][:1]

for doc in results:
    print(f"{doc.page_content} [{doc.metadata}]")  # noqa: T201  # pyright: ignore[reportUnknownMemberType]

this is surreal [{'source': 'https://surrealdb.com'}]


If you want to execute a similarity search and receive the corresponding scores you can run:


In [12]:
results = [
    (doc, score)
    for doc, score in vector_store.similarity_search_with_score(query="thud", k=4)
    if doc.metadata.get("source") == "https://surrealdb.com"
][:1]

for doc, score in results:
    print(f"[similarity={score:.0%}] {doc.page_content}")  # noqa: T201

[similarity=67%] foo


### Query by turning into retriever

You can also transform the vector store into a retriever for easier usage in your chains.


In [13]:
retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 1, "lambda_mult": 0.5}
)
_ = retriever.invoke("surreal")

## Usage for retrieval-augmented generation

For guides on how to use this vector store for retrieval-augmented generation (RAG), see the following sections:

- [How-to: Question and answer with RAG](https://python.langchain.com/docs/how_to/#qa-with-rag)
- [Retrieval conceptual docs](https://python.langchain.com/docs/concepts/retrieval/)


## Next steps

- Change `SURREAL_NAMESPACE` or `SURREAL_DATABASE` to isolate different experiments.
- Swap to a different embedding model if you want faster startup or different retrieval quality.
- Use the retriever cell above as the starting point for a local RAG prototype.
- Explore more SurrealDB resources in the [SurrealDB documentation](https://surrealdb.com/docs) and [Awesome SurrealDB](https://github.com/surrealdb/awesome-surreal).
